# Analysis Inputs
This notebook introduces the `AnalysisInput` structures and methods to plot and understand distributions.
This involves:
 - Loading pre-saved `AnalysisInput` files (not producing them here yet).
 - Accessing and plotting distributions from those files.
 - Comparing true and reconstructed distributions.
 - Investigating correlations between variables.

Each of these will be discussed in more detail throughout this notebook.

The python code will contain explainatory comments, as well as pointers to tasks you need to do.
Explanatory comments will be given as `# comments`, but tasks will use an `"""alternative comment"""` format.

In [ ]:
# Loading modules

# There is probably a nicer way to do this, but I've found this to be most reliable
# This tells python where it can find the packages that makeup this module
"""Set this path to point to the `analysis/` folder in your clone of the repository"""
ana_folder_path = "/users/wx21978/projects/pion-argon-xs-analysis/analysis"
import sys
sys.path.append(ana_folder_path) # Adds the folder to the path

# Costum modules
from python.analysis import Master
from python.analysis import cross_section as xs

# Other useful modules
import numpy as np
import awkward as ak
import matplotlib.pyplot as plt

## Loading inputs
In general, the inputs are created after each of the selection type steps have been applied.
When we produce the input files for MaCh3, we also produce the corresponding analysis input object to ensure we have access to that information.
At this point, we are not running the selection steps, so we shall load one of these pre-produced analysis input files.
This is done in the box below, using the Master.LoadObject method, which is that partner of the Master.SaveObject that was used to save the files.

In [ ]:
# Load events
ana_input_path = "/storage/0/wx21978/phys/2GeV_ana_input/Set3-2026_08_28/analysis_input_mc_selected.dill"

# Data analysis input, not looking at this quite yet
# data_ana_input_path = "/storage/0/wx21978/phys/2GeV_ana_input/Set3-2026_08_28/analysis_input_data_selected.dill"

ana_in = Master.LoadObject(ana_input_path)

### Contents of the `AnalysisInput`s

The `AnalysisInput` object is defined within `analysis/pyhon/analysis/cross_section.py`.
Feel free to take a look if you're interested, but the key features are decribed here.

`AnalysisInput` is a dataclass, whic hmeans it's key parameters are a set of "fields".
These fields store all the information required to make the MaCh3 inputs.
In addition to these fields, there are a series of methods (functions), mostly concerned with making and combiniting instances.
We shall ignore these additional methods for now, and forcus on the data content.

In [ ]:
# This prints out the fields available in the AnalysisInput
# There are slicker methods by importing dataclass,
#   but this works without requiring additional imports
list(ana_in.__dataclass_fields__.keys())

The above cell will display the list of fields.
This first list is the set of fields that are based on reconstructed information only.
These can be obtained from both simulation and real data.
Key quantities of note are marked as *KEY*.
 - `regions` are the reconstructed prediction of the type of interaction
 - `outside_tpc_reco` records if the reconstruction found an interaction within the region of interest.
 - `reco_track_length` *KEY* records the reconstructed distance the particle travelled within the TPC.
 - `reco_KE_int` *KEY* records the reconstructed energy at which the interation occurred.
 - `reco_KE_init` *KEY* records the initial energy of the particle measured by the "beam instrumentation" on the beamline before the TPC.
 - `reco_KE_ff` *KEY* records the predicted energy of the particle as it enters the "front face" of the TPC - the area in which we begin making our measurements.
 - `event_num` used to identify specific events.
 - `run` used to identify specific events - general a run is a period of time in which the detector runs with consistent conditions.
 - `sub_run` used to identify specific events.


This list is the set of "truth" fields, which we can only obtain from the simulation.
 - `inclusive_process` marks if an event was truely a pion interaction (not true for e.g. muon beam particles).
 - `exclusive_process` marks the true type of interaction that occurred.
 - `outside_tpc_true` marks if the interaction occured within the detector or not.
 - `true_track_length` *KEY* records the true distance the particle travelled within the TPC
 - `true_KE_int` *KEY* records the true energy at which the interation occurred.
 - `true_KE_init` *KEY* records the initial energy of the particle before it enters the TPC.
 - `true_KE_ff` *KEY* records the energy of the particle as it enters the "front face" of the TPC - the area in which we begin making our measurements.
  - `weights` change the relative importance of the event based on the initial energy to account for a mismatch between the simulated initial energy distribution from that which is seen in real data.

All the entries in the `AnalysisInput` are per event, so understanding the mapping between the true properties of the simulated event and what was reconstructed is simple.
In the future, we will need to consider individual PFOs (the reconstructed objects which make up the events), which will add another level of complexity to reconstruction-truth relations, for now, jsut be forewarned that not all "truth" is a simple as the examples here.

The following are unsed:
 - `process_id`
 - `region_id`

### Data format
The fields are stored as awkward arrays.
Python also has numpy arrays, which you may be familiar with.
Numpy array require the size in each dimension to be exactly specified.
Awkward array allow the sizes to be "ragged", meaning the the number of entries does not need to be consistent in each dimension.
This property will be used more when considering PFOs, but for now, these array all behave nicely as they are one dimensional.

Let's consider the true initial energy.

In [ ]:
# Good practice to avoid accidentally changing the underlying distribution stored in the analysisInput
true_init_e = ana_in.KE_init_true

# Print some values
print(true_init_e[:10])

# We will want to check statistics multiple times, so let's make a function
def check_stats(array):
    # Investigae the size of the array (since this is 1D, it is also the shape)
    print(f"Total count: {ak.count(array)}")
    
    # Basic statistics (values are MeV)
    print(f"Mean: {ak.mean(array)}")
    print(f"Standard deviation: {ak.std(array)}")
    print(f"Minimum value: {ak.min(array)}")
    print(f"Maximum value: {ak.max(array)}")

check_stats(true_init_e)

You can see from the statistics that not all the values make sense. The minimum shows some 0 MeV events.
Generally this implies that the simulated beam particle interacted or similar before it reached the detector.
Which also means that whatever was reconstructed as the beam particle was some other random particle that wasn't the real beam, since the real beam never arrived!

We don't have the tools to investigate these causes yet.
For now, let's:
 - Count how often this occurs
 - "Mask" these events out so we ignore them when looking at distributions - we're interested in the particles that actually got measured!

In [ ]:
# This creates a mask to slect these events
# Masks are array with only True or False
no_energy_event_mask = true_init_e == 0
has_energy_event_mask = np.logical_not(no_energy_event_mask) # Invert the no energy mask

# We can count how often this occurs by abusing the fact True is represented as 1 as an integer
#   False is represented as 0
# So summing the mask gives us a count of the number of `True`s present.`
print(f"Number of no energy events: {ak.sum(no_energy_event_mask)}")

# Masks can be applied to other arrays, if they have the same length
valid_true_init_e = true_init_e[has_energy_event_mask]

# Let's rheck the statistics again
check_stats(valid_true_init_e)

Great! we've removed 15 weird events out of 190641.
This is a sufficiently small fraction that we don't need to worry about impacting the overall distribution too much.

## Plotting distributions

The simplest method to view parameters of interest when dealing with large datasets is usually a histogram.
In general, the very first thing to do when dealing with any kind of quantity will be to plot it's distribution as a histogram, so you can get a feel for how it might behave.

To begin, let's try to look at the true initial energies.
Below is the matplotlib `hist` function ([information here](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html)), which produces a very simple histogram of the data you provide.
Your task will be to improve the simple function by adding the standard things you might want from a plot.
To name a few:
 - Axis labels.
 - Improved binning (general 30 bins in the region of interest is a good initial guide).
 - Specifying the range of value you show. You can investigate the very high and low energies with a mask, see how often they occur, and decide if they need to be present in this plot, or if you want to focus on just the "normal" range within the centre of the distribution. For reference, with 200,000 events, we expect to see ~4 sigma variation.

In [ ]:
"""Improve this! You will probably want to add additional arguements"""
plt.hist(valid_true_init_e)
plt.show()

## Comparing distributions

We've looked at the truth distribution of the initial energy.
Let's start asking a question: "how well do our reconstructed energies match the true energies?"
To do this, we will need to compare these qunaities.
The first vital step is to ensure we have consistency between the quantities we are comparing.
In that case, we removed a few events from the true particles when the energy was 0 MeV.
We need to ensure that these same events are removed to produce a fair comparison between our reconstructed and truth information.

In [ ]:
reco_init_e = ana_in.KE_init_reco

# Reuse the mask from before
# If you have applied any additional masks, make sure they are also included here
# np.logical_or, or np.logical_and can combine masks
valid_reco_init_e = reco_init_e[has_energy_event_mask]

# Check the counts match the truth!
check_stats(valid_reco_init_e)

"""You could investigate how often we get 0 MeV here, and decided if you want to mask these too (from both thre true and reco values)"""

As stated before, when looking at parameters, plotting the distributions is a great first step.
Let's get fancy, and put both the reconstructed distribution and new distributions on the same plot.
Again, here is skeleton code.
In addition to the previous considerations:
 - How you can make both distributions clear (hin, `histtype=step"` is nice)
 - Add a legend
 - Ensure both distributions use the same bins

You can also consider writing a function to do this comparision (it may well come in handy throughout your work, if you end up on the python side of the analysis).

In [ ]:
plt.hist(valid_true_init_e)
plt.hist(valid_reco_init_e)
plt.show()

## Correlating variables
Our distributions look similar, great.
But more than just the distributions, we want to ask "are the energies similar event-by-event?"
We can begin looking at this with more histograms!

This time, we'll use a 2D histogram.
Ideally, events will be clustered around teh positive diagonal, which implies that the true and reconstructed energies are correlated, rather than coming from two distributions which happen to look similar by chance.

Once again, here is the simplest setup.
You might consider:
 - Axis labels etc.
 - Colour bar and label
 - Colour scale (you could try a logarithmic scale)
 - Binning, always binning.
 - (Stretch goal) You could add projections of the 1D distributions at the top and side of the plot, like [this](https://matplotlib.org/stable/gallery/lines_bars_and_markers/scatter_hist.html) example.

In [ ]:
"""Improve this!"""
# awkward arrays don't seem to go well with matplotlib, so convert these to numpy arrays
plt.hist2d(valid_true_init_e.to_numpy(), valid_reco_init_e.to_numpy())
plt.show()

### Another method of comparison

Seeing the distribution over all energies is great, but it is still easier to understand 1d histograms.
So, we can effectively take a projection of the 2D histogram along the x=y direction.
This makes it easier to view the correlations, but we lose information about how the correlations depend upon the energy .In general looking at both is a good idea.

In general, you should check both the absolute error and fractional error.
Which is relevant depends on the source of and variation, in this case percfractionalentage errors are more appropriate because the errors come from a fixed spatial resolution.
The corresponding error this increases in magntiude as the energy increases.

As ever, this is a skeleton, make some improvements to the plot!

In [ ]:
# The arrays are structure such that, with the same masks applied,
#   the indicies corresponding to specific events are shared
#   between the reco and true quantities.
reco_error = valid_true_init_e - valid_reco_init_e

reco_frac_err = reco_error/valid_true_init_e

"""Improve this¬"""
plt.hist(reco_frac_err)